# SHAP Explanations on the XGBoost Model

**Week 5 · Lesson 11** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week5/lectures/11_shap_explanations_xgboost.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Adapted from ISLP Ch. 8.2 where relevant.*

## Setup

Same boilerplate as every notebook this term — see Week 1 Lesson 8 if this is your
first time seeing it.

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

## 1. Business Problem

**Business question:** for a specific customer the model flags as high-risk, can
the retention team get an honest, auditable answer to *"why does the model think
this customer will churn"* — not "tenure matters overall" (Lesson 9), but the
exact reasoning for *this* customer?

**Why it matters:** Lesson 8 built and tuned an XGBoost model on this data; Lesson 9 showed that feature importance is a global, model-specific summary;
Lesson 10 showed split-based importance can be actively misleading, and sorted
out which tool answers which question: permutation importance for what a model
needs on new data, SHAP for how it reached each prediction. This notebook
computes real SHAP values on the real tuned model — a global view of what the
model uses and, for the first time this course, an honest per-customer
explanation. It also speaks directly to the Responsible AI
discussion thread running all term: a model that
can't explain an individual decision is hard to hold accountable for one.

## 2. Data

*Load, describe, and sanity-check the data before touching a model.*

Same dataset and same split as every prior lesson that has touched churn —
**Telco Customer Churn**, `load_churn()` from `gbus564_coursework.utils.course_data`, identical
`test_size`/`random_state`/`stratify` to Lessons 3 and 8.

In [ ]:
from gbus564_coursework.utils.course_data import load_churn

df = load_churn()
df.head()

In [ ]:
display(df.describe())

display(df.describe(exclude="number"))

### Data Prep

Same one-hot-encoding-only prep as every tree-based lesson this week — trees
don't need scaling (Lesson 3).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

X, y = df.drop(columns="Churn"), df["Churn"]
categorical_cols = X.select_dtypes(exclude="number").columns

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

preprocess = ColumnTransformer(
    [("encode", OneHotEncoder(handle_unknown="ignore"), categorical_cols)],
    remainder="passthrough",
).fit(X_train)

feature_names = [n.replace("encode__", "").replace("remainder__", "")
                  for n in preprocess.get_feature_names_out()]
X_train_enc = pd.DataFrame(preprocess.transform(X_train), columns=feature_names, index=X_train.index)
X_test_enc = pd.DataFrame(preprocess.transform(X_test), columns=feature_names, index=X_test.index)

## 3. Model

*Baseline first, then the model this notebook explains.*

The tuned XGBoost from Lesson 8 — these are that notebook's randomized-search
winners (with early stopping's 128 rounds as `n_estimators`), not re-derived
here, the same way Lesson 8 reused Lesson 3's
`ccp_alpha`. SHAP's `TreeExplainer` (below) reads a fitted tree model directly,
so the classifier is fit on the already-encoded data rather than wrapped in a
`Pipeline` this time. Not part of the shared Setup cell (Lesson 8's note still
applies) — installed fresh here too, since this notebook runs standalone.

In [ ]:
%pip install -q xgboost
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier

baseline = DummyClassifier(strategy="most_frequent").fit(X_train_enc, y_train)

xgboost_model = XGBClassifier(
    n_estimators=128, max_depth=3, learning_rate=0.05, subsample=0.6,
    colsample_bynode=0.6, gamma=0, reg_lambda=1, reg_alpha=1, min_child_weight=1,
    random_state=RANDOM_SEED,
).fit(X_train_enc, y_train)

## 4. Evaluate

*Confirm this is a model worth explaining before explaining it.*

Lesson 8 already ran the full model comparison and tuning search — this is
just a quick confirmation on this exact fit, not a repeat of that work.

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score

baseline_acc = accuracy_score(y_test, baseline.predict(X_test_enc))
xgb_acc = accuracy_score(y_test, xgboost_model.predict(X_test_enc))
xgb_auc = roc_auc_score(y_test, xgboost_model.predict_proba(X_test_enc)[:, 1])

print(f"Baseline accuracy: {baseline_acc:.2%}")
print(f"XGBoost accuracy:  {xgb_acc:.2%}   AUC: {xgb_auc:.3f}")

## 5. Interpret

*What is the model actually keying off of — globally, and for one customer?*

Lesson 8's `feature_importances_` charts and Lesson 10's demonstration both showed
why that number can mislead. `TreeExplainer` computes exact SHAP values for this
model — fast, because XGBoost trees have a closed-form solution (TreeSHAP), not
an approximation.

In [ ]:
%pip install -q shap
import shap

explainer = shap.TreeExplainer(xgboost_model)
shap_values = explainer(X_test_enc)
shap_values.shape

### Global: the honest version of Lesson 8's importance chart

Averaging `|SHAP value|` across every test customer gives a global ranking —
built from real per-customer accounting instead of training-time impurity
shortcuts. Color shows each customer's actual feature value, so the plot also
shows *direction*: a cluster of red (high value) points on the positive side
means high values of that feature push toward churn.

**How to read it:**

- **Rows** — one per feature, sorted by average `|SHAP value|` across the test
  customers (absolute value first, so a feature that pushes some customers up
  and others down doesn't average out to zero). Top row = what the model leans
  on most.
- **Each dot** — one test customer. Every row has a dot for every customer.
- **Horizontal position** — how far that feature *moved* that customer's
  prediction away from the base value, the model's average prediction (`0` on
  the axis). Right of `0` pushes toward churn, left pulls away from it. It's a
  movement from the baseline, not a prediction, so a dot at `0` means "this
  feature didn't change this customer's score," not "zero risk."
- **Color** — that customer's actual value of the feature: red is high, blue is
  low (for a one-hot column, red = 1 and blue = 0). Sign plus color together give
  the relationship: blue `tenure` dots on the right means *short* tenure pushes
  toward churn.
- **Spread** — a wide row means the feature's push varies a lot from customer
  to customer; a tight clump near `0` means it rarely moves anyone.

**The units are log-odds, not probability.** `TreeExplainer` explains
XGBoost's raw output, the same log-odds scale as Week 3 Lesson 3's logistic
regression coefficients (the waterfall below spells this out). Log-odds are
unbounded, so SHAP values aren't capped at ±1. And because the sigmoid is
curved, the same push changes the probability more for a customer near 50%
than for one near 5%. Read the x-axis as "bigger or smaller push," not as
percentage points.

In [ ]:
shap.plots.beeswarm(shap_values, max_display=10)

`Contract_Month-to-month` still leads, same as Lesson 8's XGBoost importance
chart — but now every other feature's ranking comes with a directional story
attached (`tenure`: low values, in blue, sit on the churn-pushing side; `MonthlyCharges`:
high values, in red, do too), not just a bar with no sign.

### Local: one customer, one honest answer

Pick the single highest-risk customer in the test set and ask SHAP the question
Lesson 9 said a global importance chart cannot answer.

In [ ]:
proba = xgboost_model.predict_proba(X_test_enc)[:, 1]
top_customer = int(np.argmax(proba))

print(f"Predicted churn probability: {proba[top_customer]:.1%}")
print(f"Average customer's churn rate: {y_train.mean():.1%}")
X_test.iloc[top_customer][["tenure", "Contract", "InternetService", "MonthlyCharges", "TechSupport"]]

In [ ]:
shap.plots.waterfall(shap_values[top_customer])

Read the axis: SHAP values here are on the **log-odds scale** — the same scale
Week 3 Lesson 3's logistic regression coefficients live on, not a direct
probability. `E[f(X)]` is the average customer's log-odds of churn; `f(x)` at
the top is this customer's log-odds, after every feature's contribution is
added. Passing that final number through the sigmoid (Week 3's log-odds →
probability step) is exactly what turns it into the predicted-probability
figure printed above — the two numbers are the same fact, on two different
scales. For this specific customer — one month of tenure, paying $85.05/month
for fiber-optic internet on a month-to-month contract — the single largest
push toward churn is that one-month tenure itself, ahead of the contract type
that leads the global ranking, with the low total charges third (just one
month's bill so far — the same "brand-new customer" signal, seen through a
second column): not the story
Lesson 8's aggregate importance chart alone would have led anyone to expect,
and exactly the per-customer honesty a global ranking cannot provide.

## 6. Recommend

**Recommendation:** attach a SHAP waterfall like the one above to every
high-risk flag the retention team acts on — not as a nice-to-have, but as the
actual justification for the outreach. "Why is this customer flagged" now has
an honest, additive answer instead of a borrowed global importance chart or no
answer at all.

**Connecting to Responsible AI:** transparency and accountability are threads
this course returns to all term through the weekly discussion readings, not a
single lecture — this notebook is where the mechanics catch up to that
question. One caveat matters more than the mechanics: **SHAP explains what the
model is doing, not whether the model should be trusted.** If
`Contract_Month-to-month` or a correlate of a protected characteristic were
doing the work for the wrong reason — encoding historical bias rather than
genuine risk — SHAP would explain that reasoning just as fluently as it
explains a legitimate one. An honest explanation is a prerequisite for
auditing a model's fairness, not a substitute for actually doing that audit.

**Where this week ends:** ten lessons ago this week opened with a single
splitting rule (Lesson 1). It closes here, with an honest, per-customer account
of a tuned ensemble of hundreds of those rules — the same throughline
(transparency, then accuracy, then transparency again, now for a more powerful
model) that Weeks 3 and 4 also followed, and Week 6 will follow once more.